In [23]:
from typing import TypedDict, Literal

class Ingredient(TypedDict):
    name: str
    amount: float
    unit: str
    flavour: Literal[
        'acidity', 'salt', 'spice', 'bitter', 
        'creamy', 'sweet', 'none', 'aromatic'
        ]
    type: Literal[
        'starches', 'sugars', 'salts', 'fats', 
        'seeds_nuts_dried_fruits', 'binders', 
        'produce', 'spices', 'sugars', 'liquids'
        ]
    ingredients: list[Ingredient]
    cons: Literal["SOLID", "LIQUID"]


class Recipe(TypedDict):
    name: str
    servings: int
    ingredients: list[Ingredient]
    source: str

In [24]:
import json
from typing import Dict, List, Any, Literal
from load_json import load_json_file
import pandas as pd
initial_dataset = []
paths:list[str] = [
    '../data/western_cookies.json',
    '../data/indonesian_baked_goods.json',
    '../data/indonesian_drinks.json'
]

for path in paths:
    data:list[Dict[unknown, Unknown]] = load_json_file(path)
    initial_dataset.extend(data)

def find_and_replace_recipe_properties(
    inspiring_set: list[Recipe],
    replacement_data: pd.DataFrame,
    target_properties: list[str] | None = None,
    reference_column: str = "ingredient",
) -> list[Recipe]:
    properties = target_properties or ["flavour", "type", "is_indonesian"]

    if reference_column not in replacement_data.columns:
        raise KeyError(
            f"Column {reference_column!r} not found in metadata. "
            f"Available columns: {list(replacement_data.columns)}"
        )

    for recipe in inspiring_set:
        for ingredient in recipe["ingredients"]:
            ingredient_name = ingredient.get("name")
            if not ingredient_name:
                continue

            matching_rows = replacement_data[
                replacement_data[reference_column] == ingredient_name
            ]

            if matching_rows.empty:
                continue

            row = matching_rows.iloc[0]
            for prop in properties:
                if prop in matching_rows.columns and pd.notna(row[prop]):
                    value = row[prop]
                    ingredient[prop] = value.item() if hasattr(value, 'item') else value

    return inspiring_set

def norm_data(recipes:list[Recipe]) -> list[Recipe]:
    for recipe in recipes:
        for i in recipe["ingredients"]:
            i["source"]=recipe["name"]
            if i["unit"] in ["tsp", "tsps"]:
                i["unit"]="g" if i["cons"]=="SOLID" else "ml"
                i["amount"]=i["amount"]*5.0

            elif i["unit"] in ["Tbsp", "Tbsps"]:
                i["unit"]="g" if i["cons"]=="SOLID" else "ml"
                i["amount"]=i["amount"]*15.0
            elif i["unit"]=="pkt":
                i["unit"]="g"
                i["amount"]=i["amount"]*16
            elif i["unit"]=="pinch":
                i["unit"]="g" if i["cons"]=="SOLID" else "ml"
                i["amount"]=i["amount"]*0.5
            elif i["unit"] in ["stick", "sticks"]:
                i["unit"]="g"
                i["amount"]=i["amount"]*113
            elif i["name"] in ["eggs", "egg", "egg yolk", "egg yolks"]:
                if i["unit"]=="large":
                    i["amount"] = i["amount"] * 55.0
                else:
                    i["amount"] = i["amount"] * 50.0
                i["unit"]="g"
            elif i["unit"] in ["cup", "cups"]:
                if i["cons"] == "LIQUID":
                        i["amount"] = i["amount"] * 240.0
                        i["unit"] = "ml"
                else:
                    i["amount"] = i["amount"] * 130.0
                    i["unit"] = "g"
    return recipes

print(initial_dataset)
file = open("inspiring_set.json", 'w')
file.write(json.dumps(norm_data(initial_dataset)))
file.close()


[{'name': 'Sugar Cookies', 'family': 'cookie', 'servings': 16, 'ingredients': [{'name': 'flour', 'amount': 500.0, 'unit': 'g', 'type': 'Baking', 'cons': 'SOLID'}, {'name': 'salt', 'amount': 0.5, 'unit': 'tsps', 'type': 'Spices and Seasonings', 'cons': 'SOLID'}, {'name': 'baking powder', 'amount': 1.0, 'unit': 'tsp', 'type': 'Baking', 'cons': 'SOLID'}, {'name': 'butter', 'amount': 227.0, 'unit': 'g', 'type': 'Milk, Eggs, Other Dairy', 'cons': 'SOLID'}, {'name': 'sugar', 'amount': 400.0, 'unit': 'g', 'type': 'Baking', 'cons': 'SOLID'}, {'name': 'eggs', 'amount': 2.0, 'unit': 'large', 'type': 'Milk, Eggs, Other Dairy', 'cons': 'SOLID'}, {'name': 'lemon juice', 'amount': 2.0, 'unit': 'tsps', 'type': 'Baking', 'cons': 'LIQUID'}, {'name': 'lemon zest', 'amount': 2.0, 'unit': '', 'type': 'Produce', 'cons': 'SOLID'}]}, {'name': 'Muesli Cookies', 'family': 'cookie', 'servings': 8, 'ingredients': [{'name': 'flour', 'amount': 180.0, 'unit': 'g', 'type': 'Baking', 'cons': 'SOLID'}, {'name': 'muesl

In [25]:

inspiring_set:list[Dict[Unknown, Unknown]] = []

paths:list[str] = [
    # '../data/western_cookies.json',
    # '../data/indonesian_baked_goods.json',
    # '../data/indonesian_drinks.json'
    '../data/inspiring_set.json'
]

for path in paths:
    data:list[Dict[Unknown, Unknown]] = load_json_file(path)
    inspiring_set.extend(data)

print(str(inspiring_set))

[{'name': 'Jamu Kunyit Asam (Cook Me Indonesian)', 'family': 'kunyit_asam', 'servings': 8, 'source': 'https://www.cookmeindonesian.com', 'ingredients': [{'name': 'fresh turmeric', 'amount': 300, 'unit': 'g', 'flavour': 'spice', 'type': 'spices', 'cons': 'SOLID', 'is_indonesian': True, 'source': 'Jamu Kunyit Asam (Cook Me Indonesian)'}, {'name': 'tamarind', 'amount': 50, 'unit': 'g', 'flavour': 'acidity', 'type': 'produce', 'cons': 'SOLID', 'is_indonesian': True, 'source': 'Jamu Kunyit Asam (Cook Me Indonesian)'}, {'name': 'palm sugar', 'amount': 150, 'unit': 'g', 'flavour': 'sweet', 'type': 'sugars', 'cons': 'SOLID', 'is_indonesian': True, 'source': 'Jamu Kunyit Asam (Cook Me Indonesian)'}, {'name': 'water', 'amount': 2000, 'unit': 'ml', 'flavour': 'none', 'type': 'liquids', 'cons': 'LIQUID', 'is_indonesian': False, 'source': 'Jamu Kunyit Asam (Cook Me Indonesian)'}], 'fitness': 1}, {'name': 'Jamu Kunyit (Cookpad)', 'family': 'kunyit_asam', 'servings': 1, 'source': 'https://cookpad.com

In [26]:
def get_unique_values(key:str) -> set:
    return {
        ingredient[key]
        for recipe in inspiring_set
        for ingredient in recipe["ingredients"]
        if key in ingredient
    }

unique_flavours = get_unique_values("flavour")
unique_types = get_unique_values("type")
unique_ingredients = get_unique_values("name")
unique_families = {recipe["family"] for recipe in inspiring_set}

print(unique_flavours)
print(unique_types)
print(unique_ingredients)
print(unique_families)

{'none', 'aromatic', 'bitter', 'salt', 'acidity', 'creamy', 'spice', 'sweet'}
{'leaving_agents', 'fats', 'binders', 'produce', 'seeds_nuts_dried_fruits', 'sugars', 'liquids', 'salts', 'spices', 'starches'}
{'spectrum all-vegetable shortening', 'egg whites', 'nestlé® toll house® premier morsels', 'palm sugar', 'lemon', 'vanilla', 'crunchy peanut butter', 'baking power', 'jackfruit', 'chocolate chips - chocolate or', 'blackstrap molasses', 'banana leaf', 'glutinous rice flour', 'vanilla bean', 'roll of pillsbury chocolate chips cookies', 'ground allspice', 'orange zest', 'apple sauce', 'pecans', 'baking cocoa', 'honey/maple syrup', 'sago starch', 'bisquick mix', 'egg white', 'chocolate', 'slivered almonds', 'half and half', 'vegan chocolate chips and/or pieces', 'chocolate bar', 'egg yolks', 'cream cheese', 'roasted peanuts', 'ap flour', 'duncan hines classic decadent cake mix', 'raisins', 'flaky sea salt', 'almond flakes on top as decoration', 'vegetable oil', "confectioners' sugar", 'm

In [27]:
%pip install scipy
from typing import TypedDict
import pprint
import numpy as np
from scipy import stats
recipe_flavour_ratios = []

# def get_ingredient_ratios(recipe: Recipe, key: str) -> float:
#     total_recipe_amount = sum(ing["amount"] for ing in recipe["ingredients"])
#     amounts_per_category = {}
#     for ingredient in recipe["ingredients"]:
#         category = ingredient[key]
#         amounts_per_category[category] =+ ingredient["amount"]

#     for category, amount in amounts_per_category.items():
#         amounts_per_category[category] = amount / total_recipe_amount if total_recipe_amount > 0 else 0

#     return amounts_per_category

def get_ingredient_ratios(recipe: Dict[str, Any], key: str) -> Dict[str, float]:
    total_recipe_amount = sum(
        ingredient["amount"] for ingredient in recipe["ingredients"]
    )

    amounts_per_category: Dict[str, float] = {}

    for ingredient in recipe["ingredients"]:
        if key not in ingredient:
            continue
        category = str(ingredient[key])
        amounts_per_category[category] = (
            amounts_per_category.get(category, 0) + ingredient["amount"]
        )

    return {
        category: amount / total_recipe_amount if total_recipe_amount > 0 else 0
        for category, amount in amounts_per_category.items()
    }

class RatioResult(TypedDict):
    name: str
    ratios: dict[str, float]


class CategoryStatistics(TypedDict):
    mean: float
    min: float
    max: float


def calculate_category_statistics(
    recipes: list[dict[str, Any]],
    category_key: str,
) -> tuple[list[RatioResult], dict[str, CategoryStatistics]]:
    categories: set[str] = {
        str(ingredient[category_key])
        for recipe in recipes
        for ingredient in recipe["ingredients"]
        if category_key in ingredient
    }

    recipe_ratios: list[RatioResult] = []

    for recipe in recipes:
        ratios = get_ingredient_ratios(recipe, category_key)

        complete_ratios = {
            category: ratios.get(category, 0.0)
            for category in categories
        }

        recipe_ratios.append({
            "name": str(recipe["name"]),
            "ratios": complete_ratios,
        })

    statistics: dict[str, CategoryStatistics] = {
        category: {
            "mean": sum(
                item["ratios"][category] for item in recipe_ratios
            ) / len(recipe_ratios),
            "min": min(item["ratios"][category] for item in recipe_ratios),
            "max": max(item["ratios"][category] for item in recipe_ratios),
        }
        for category in categories
    }

    return recipe_ratios, statistics


recipe_flavour_ratios, flavour_statistics = calculate_category_statistics(
    inspiring_set,
    "flavour",
)

recipe_type_ratios, type_statistics = calculate_category_statistics(
    inspiring_set,
    "type",
)

# print("Flavour Statistics:", pprint.pformat(flavour_statistics))
print("Type Statistics:", pprint.pformat(type_statistics))

Note: you may need to restart the kernel to use updated packages.
Type Statistics: {'binders': {'max': 1.0, 'mean': 0.46685928226342177, 'min': 0.0},
 'fats': {'max': 0.4860813704496788, 'mean': 0.046108962474646686, 'min': 0.0},
 'leaving_agents': {'max': 0.13855098536649355,
                    'mean': 0.0014688651222380359,
                    'min': 0.0},
 'liquids': {'max': 1.0, 'mean': 0.21712010361733802, 'min': 0.0},
 'produce': {'max': 0.5241090146750524,
             'mean': 0.03358920056905473,
             'min': 0.0},
 'salts': {'max': 0.008928571428571428,
           'mean': 0.0007245710188543671,
           'min': 0.0},
 'seeds_nuts_dried_fruits': {'max': 0.7292636867275065,
                             'mean': 0.0421045780793713,
                             'min': 0.0},
 'spices': {'max': 0.12, 'mean': 0.011020763027162695, 'min': 0.0},
 'starches': {'max': 0.6535947712418301,
              'mean': 0.0863374556831019,
              'min': 0.0},
 'sugars': {'max': 0.954